# 实时采样质量复核
逐关节、逐帧核对有效率与位置散布。仅复核已采样输出，不将散布解释为准确度。操作者确认未静止的段不用于静止试验。

In [ ]:
from pathlib import Path
import csv, json
import numpy as np
base = Path.cwd()
if base.name != 'trajectory_validation_v1':
    base = base / '04_实验设计/D435i_感知验证/TP-D435I-01/trajectory_validation_v1'
for summary_path in sorted(base.glob('trajectory_*/summary.json')):
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    rows = list(csv.DictReader((Path(summary['raw_directory']) / 'keypoints.csv').open(encoding='utf-8')))
    keys = [(r['frame_id'], r['host_receive_mono_s'], r['joint_id']) for r in rows]
    assert len(keys) == len(set(keys)), 'duplicate key'
    for metric in summary['joints']:
        joint_rows = [r for r in rows if r['joint_id'] == metric['joint_id']]
        valid = [r for r in joint_rows if r['measurement_valid'] == 'True']
        assert len(joint_rows) == summary['unique_outputs']
        assert len(valid) == metric['valid_count']
        a = np.array([[float(r[k]) for k in ['x_m','y_m','z_m']] for r in valid])
        if len(a) > 1:
            sd = a.std(axis=0, ddof=1) * 1000
            p95 = np.percentile(np.linalg.norm(a - np.median(a, axis=0), axis=1)*1000, 95)
            assert np.allclose(sd, metric['axis_sd_mm'])
            assert np.isclose(p95, metric['median_distance_p95_mm'])
        assert metric['accuracy_status'] == 'BLOCKED_NO_INDEPENDENT_REFERENCE'
    print(summary['run_id'], 'CSV metrics checked; independent accuracy reference absent')
